## TodoListMiddleware
- 强制把计划挂在全局状态里，时刻提醒它“下一步该干什么”,适合长任务,防止模型跑偏

In [2]:
from langchain.agents.middleware import TodoListMiddleware
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage
from langchain.tools import tool
from pathlib import Path
import subprocess
WORKSPACE = Path("../todo_workspace")

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    profile={"max_input_tokens": 128_000}
)


@tool
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
    path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()
    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"
    if not target.exists():
        return f"错误：目录不存在: {path}"
    if not target.is_dir():
        return f"错误：不是目录: {path}"
    items = sorted(target.iterdir(), key=lambda p: (p.is_file(),
    p.name.lower()))
    if not items:
        return f"目录为空: {path}"
    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")
    return "\n".join(lines)

@tool
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。
    Args:
    path: 工作区内的文件名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    return file_path.read_text(encoding="utf-8")

@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。
    Args:
    path: 工作区内的文件名
    content: 写入文件的内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"

@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。
    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败: {e}"

agent = create_agent(
    model = model,
    tools=[list_files, read_file, write_file, run_tests],
    middleware=[
        #作为工具传入,参数:system_prompt:什么时候该用、以及怎么用,tool_description:工具描述,可不写有默认值
        TodoListMiddleware()
    ]
)

response = agent.invoke({
    "messages": [HumanMessage(content= "你是一个代码修复助手。遇到多步骤任务时，先使用 write_todos 制定待办事项；"
"然后读取文件、修复代码并运行测试。工作全部在工作区下进行")]
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

你是一个代码修复助手。遇到多步骤任务时，先使用 write_todos 制定待办事项；然后读取文件、修复代码并运行测试。工作全部在工作区下进行
================================== Ai Message ==================================

我先查看工作区的文件结构，了解项目情况。
Tool Calls:
  list_files (call_00_HoWzWNI8OBCXbETcaGqx6479)
 Call ID: call_00_HoWzWNI8OBCXbETcaGqx6479
  Args:
    path: .
================================= Tool Message =================================
Name: list_files

[FILE] my_add.py
[FILE] test_my_add.py
================================== Ai Message ==================================
Tool Calls:
  read_file (call_00_MKbCSAVWWMTi7nfd6vy60014)
 Call ID: call_00_MKbCSAVWWMTi7nfd6vy60014
  Args:
    path: my_add.py
  read_file (call_01_GeNBzQHfkTBJv9yPusV56306)
 Call ID: call_01_GeNBzQHfkTBJv9yPusV56306
  Args:
    path: test_my_add.py
================================= Tool Message =================================
Name: read_file

def add(a: int, b: int) -> int:
    """返回两个整数的和"""